# 09 — Narrow Population: K Dwarfs (Leakage-Controlled)

## Research question
Within the catalog's **small-planet (1–4 Earth radii), K-dwarf-temperature (3,900–5,200 K) KOI subset**, how does host-star-grouped, out-of-fold reliability scoring describe the set of catalog-confirmed planet entries?

This notebook repeats the analysis on the narrower stellar/planet-radius subset, while correcting the labeling, validation, and feature-selection problems in the original version.

### Important scientific interpretation
The denominator is the number of **unique host stars represented in this filtered KOI catalog subset**, not all K dwarfs in the Kepler target population. Therefore, the baseline and screened quantities below are **catalog-host fractions**, not intrinsic planet occurrence rates.

The “screened” value is a diagnostic formed by retaining catalog-confirmed entries whose held-out model probability exceeds a chosen threshold. It is not a corrected occurrence rate, and it does not establish that any catalog disposition is physically correct.

Candidates are excluded from supervised training and evaluation because their outcomes are unresolved. Disposition-derived scores and vetting flags are excluded from model features.

In [4]:
# 1. Imports and reproducible settings

import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
N_SPLITS = 5
K_DWARF_TEMPERATURE_MIN = 3900
K_DWARF_TEMPERATURE_MAX = 5200
PLANET_RADIUS_MIN = 1.0
PLANET_RADIUS_MAX = 4.0
SCREENING_THRESHOLD = 0.50
THRESHOLDS = [0.30, 0.50, 0.70]

# Kaggle dataset path from the original project; fallback searches /kaggle/input.
DATA_PATH = "/kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv"

if not os.path.exists(DATA_PATH):
    csv_matches = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if "cumulative" in os.path.basename(p).lower()
    )
    if not csv_matches:
        raise FileNotFoundError(
            "Could not find the KOI cumulative CSV under /kaggle/input. "
            "Set DATA_PATH manually to the attached dataset."
        )
    if len(csv_matches) > 1:
        print("Multiple cumulative CSV files found:")
        for p in csv_matches:
            print(" -", p)
        print("\nUsing the first sorted match. Change DATA_PATH if needed.")
    DATA_PATH = csv_matches[0]

print("Dataset:", DATA_PATH)

Dataset: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv


In [5]:
# 2. Load data and apply the original narrow-population filters

raw = pd.read_csv(DATA_PATH, comment="#")
print("Original catalog rows:", len(raw))

required_cols = {"koi_disposition", "kepid", "koi_prad", "koi_steff"}
missing = required_cols - set(raw.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

# Retain the dispositions present in the original catalog analysis, then apply
# the original small-planet and K-dwarf temperature cuts.
population = raw.loc[
    raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
].copy()

population = population.loc[
    population["koi_prad"].between(PLANET_RADIUS_MIN, PLANET_RADIUS_MAX, inclusive="both")
    & population["koi_steff"].between(
        K_DWARF_TEMPERATURE_MIN, K_DWARF_TEMPERATURE_MAX, inclusive="both"
    )
].copy()

population = population.dropna(subset=["kepid", "koi_prad", "koi_steff"])
population["kepid"] = population["kepid"].astype(str)

print("Filtered rows including candidates:", len(population))
print("Unique host stars including candidates:", population["kepid"].nunique())
print("Disposition counts in filtered population:")
display(population["koi_disposition"].value_counts(dropna=False).rename("rows").to_frame())

# Candidates are not labeled as false positives. Use only resolved labels for
# supervised modeling and for the directly comparable catalog-host fraction.
df = population.loc[
    population["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy().reset_index(drop=True)

df["label_confirmed"] = (df["koi_disposition"] == "CONFIRMED").astype(int)
df["host_group"] = df["kepid"]

n_rows = len(df)
n_hosts = df["host_group"].nunique()
n_confirmed_rows = int(df["label_confirmed"].sum())
n_false_positive_rows = int((df["label_confirmed"] == 0).sum())
n_candidate_rows_excluded = int((population["koi_disposition"] == "CANDIDATE").sum())

if n_hosts < N_SPLITS:
    raise ValueError(
        f"Need at least {N_SPLITS} unique resolved host stars; found {n_hosts}."
    )
if df["label_confirmed"].nunique() != 2:
    raise ValueError("Both CONFIRMED and FALSE POSITIVE rows are required.")

print("\nResolved modeling sample")
print("Rows:", n_rows)
print("Unique host stars:", n_hosts)
print("Confirmed rows:", n_confirmed_rows)
print("False-positive rows:", n_false_positive_rows)
print("Candidate rows excluded from modeling:", n_candidate_rows_excluded)

Original catalog rows: 9564
Filtered rows including candidates: 852
Unique host stars including candidates: 665
Disposition counts in filtered population:


,rows
koi_disposition,
CONFIRMED,507
CANDIDATE,176
FALSE POSITIVE,169



Resolved modeling sample
Rows: 676
Unique host stars: 520
Confirmed rows: 507
False-positive rows: 169
Candidate rows excluded from modeling: 176


## 3. Define the catalog-host fraction

For this notebook, a host is counted as a catalog-confirmed host if at least one row for that host is labeled `CONFIRMED`.

The denominator is the unique host stars with at least one **resolved** KOI row in this filtered catalog subset. This is a catalog-based descriptive denominator, not the full Kepler K-dwarf target-star population.

In [6]:
# Host-level catalog summary (descriptive only).
host_label = (
    df.groupby("host_group")["label_confirmed"]
      .max()
      .rename("has_catalog_confirmed_entry")
      .reset_index()
)

baseline_catalog_host_fraction = host_label["has_catalog_confirmed_entry"].mean()
confirmed_hosts = int(host_label["has_catalog_confirmed_entry"].sum())

print(f"Resolved-sample host stars: {n_hosts}")
print(f"Hosts with >=1 catalog-confirmed entry: {confirmed_hosts}")
print(f"Baseline catalog-host fraction: {baseline_catalog_host_fraction:.4f}")
print("\nDo not describe this as intrinsic planet occurrence.")

Resolved-sample host stars: 520
Hosts with >=1 catalog-confirmed entry: 353
Baseline catalog-host fraction: 0.6788

Do not describe this as intrinsic planet occurrence.


## 4. Feature selection and leakage controls

Use an explicit allow-list of numeric physical/observational features. Do not use catalog disposition, `koi_score`, Robovetter false-positive flags, or identifiers as predictors.

Missing values are imputed **inside each training fold** by the scikit-learn pipeline. No full-dataset median imputation is performed before cross-validation.

In [7]:
candidate_features = [
    "koi_period",
    "koi_time0bk",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_smet",
    "koi_srad",
    "koi_smass",
    "koi_kepmag",
    "koi_ror",
    "koi_max_mult_ev",
    "koi_num_transits"
]

feature_cols = [
    col for col in candidate_features
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
]

if len(feature_cols) < 3:
    raise ValueError(
        "Fewer than 3 approved numeric features found. "
        f"Available features: {feature_cols}"
    )

forbidden_tokens = [
    "disposition", "koi_score", "fpflag", "comment",
    "kepid", "kic", "label", "group"
]
leakage_hits = [
    col for col in feature_cols
    if any(token in col.lower() for token in forbidden_tokens)
]
if leakage_hits:
    raise AssertionError(f"Potential leakage columns found: {leakage_hits}")

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
y = df["label_confirmed"].astype(int)
groups = df["host_group"]

print("Features used:")
for col in feature_cols:
    print(" -", col)
print("Feature count:", len(feature_cols))

Features used:
 - koi_period
 - koi_time0bk
 - koi_impact
 - koi_duration
 - koi_depth
 - koi_prad
 - koi_teq
 - koi_insol
 - koi_model_snr
 - koi_steff
 - koi_slogg
 - koi_srad
 - koi_kepmag
Feature count: 13


## 5. Host-star-grouped out-of-fold predictions

`GroupKFold` ensures all KOI entries belonging to the same host star stay in one fold. Every resolved row is predicted exactly once by a model that was not trained on that host star.

The imputer and classifier are fitted separately within each fold.

In [8]:
n_splits = min(N_SPLITS, groups.nunique())
gkf = GroupKFold(n_splits=n_splits)

oof_probability = np.full(len(df), np.nan, dtype=float)
oof_fold = np.full(len(df), -1, dtype=int)
prediction_count = np.zeros(len(df), dtype=int)
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    train_groups = set(groups.iloc[train_idx].unique())
    test_groups = set(groups.iloc[test_idx].unique())

    overlap = train_groups.intersection(test_groups)
    if overlap:
        raise AssertionError(
            f"Host-star overlap in fold {fold}: {len(overlap)} groups."
        )
    if y.iloc[train_idx].nunique() != 2:
        raise ValueError(f"Training fold {fold} contains only one target class.")

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE + fold,
            n_jobs=-1
        ))
    ])

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    positive_position = list(model.named_steps["classifier"].classes_).index(1)
    fold_probabilities = model.predict_proba(X.iloc[test_idx])[:, positive_position]

    oof_probability[test_idx] = fold_probabilities
    oof_fold[test_idx] = fold
    prediction_count[test_idx] += 1

    fold_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_host_stars": len(train_groups),
        "test_host_stars": len(test_groups),
        "confirmed_test_rows": int(y.iloc[test_idx].sum()),
        "false_positive_test_rows": int((y.iloc[test_idx] == 0).sum())
    })

if not np.all(prediction_count == 1):
    values, counts = np.unique(prediction_count, return_counts=True)
    raise AssertionError(
        f"Each row must receive exactly one OOF prediction. "
        f"Observed counts: {dict(zip(values, counts))}"
    )
if np.isnan(oof_probability).any() or (oof_fold < 1).any():
    raise AssertionError("Missing OOF probabilities or fold assignments.")

df["oof_confirmed_probability"] = oof_probability
df["oof_fold"] = oof_fold

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

print(f"OOF coverage: {(prediction_count == 1).sum()} / {len(df)} rows")
print("All host-star groups were held out intact.")

,fold,train_rows,test_rows,train_host_stars,test_host_stars,confirmed_test_rows,false_positive_test_rows
0,1,540,136,416,104,106,30
1,2,541,135,416,104,104,31
2,3,541,135,416,104,92,43
3,4,541,135,416,104,104,31
4,5,541,135,416,104,101,34


OOF coverage: 676 / 676 rows
All host-star groups were held out intact.


## 6. Pooled out-of-fold classifier performance

These metrics describe discrimination between catalog-confirmed and catalog-false-positive entries in the resolved filtered sample. They do not validate the status of unresolved candidates.

In [9]:
pred_at_threshold = (oof_probability >= SCREENING_THRESHOLD).astype(int)

pooled_metrics = {
    "n_resolved_rows": len(df),
    "n_resolved_host_stars": int(groups.nunique()),
    "n_confirmed_rows": int((y == 1).sum()),
    "n_false_positive_rows": int((y == 0).sum()),
    "n_features": len(feature_cols),
    "n_group_folds": n_splits,
    "oof_roc_auc": roc_auc_score(y, oof_probability),
    "oof_average_precision": average_precision_score(y, oof_probability),
    f"balanced_accuracy_at_{SCREENING_THRESHOLD:.2f}": balanced_accuracy_score(y, pred_at_threshold),
    f"precision_at_{SCREENING_THRESHOLD:.2f}": precision_score(y, pred_at_threshold, zero_division=0),
    f"confirmed_recall_at_{SCREENING_THRESHOLD:.2f}": recall_score(y, pred_at_threshold, zero_division=0),
    f"f1_at_{SCREENING_THRESHOLD:.2f}": f1_score(y, pred_at_threshold, zero_division=0)
}

metrics_df = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in pooled_metrics.items()]
)
display(metrics_df)

print("Confusion matrix at threshold", SCREENING_THRESHOLD)
print("Rows = actual [FALSE POSITIVE, CONFIRMED]")
print("Cols = predicted [FALSE POSITIVE, CONFIRMED]")
print(confusion_matrix(y, pred_at_threshold, labels=[0, 1]))

print("\nClassification report")
print(classification_report(
    y, pred_at_threshold,
    labels=[0, 1],
    target_names=["FALSE POSITIVE", "CONFIRMED"],
    zero_division=0
))

,metric,value
0,n_resolved_rows,676.000000
1,n_resolved_host_stars,520.000000
2,n_confirmed_rows,507.000000
3,n_false_positive_rows,169.000000
4,n_features,13.000000
5,n_group_folds,5.000000
6,oof_roc_auc,0.918134
7,oof_average_precision,0.956868
8,balanced_accuracy_at_0.50,0.834320
9,precision_at_0.50,0.913295


Confusion matrix at threshold 0.5
Rows = actual [FALSE POSITIVE, CONFIRMED]
Cols = predicted [FALSE POSITIVE, CONFIRMED]
[[124  45]
 [ 33 474]]

Classification report
                precision    recall  f1-score   support

FALSE POSITIVE       0.79      0.73      0.76       169
     CONFIRMED       0.91      0.93      0.92       507

      accuracy                           0.88       676
     macro avg       0.85      0.83      0.84       676
  weighted avg       0.88      0.88      0.88       676



## 7. Threshold-based screened catalog-host fraction

At each threshold, a host is counted in the screened numerator if at least one of its **catalog-confirmed** entries has a grouped OOF confirmed-class probability at or above that threshold.

This is a sensitivity analysis of a catalog-derived statistic. It is not a revised intrinsic occurrence rate.

In [10]:
df["is_high_confidence_at_threshold"] = (
    df["oof_confirmed_probability"] >= SCREENING_THRESHOLD
)

threshold_results = []
for threshold in THRESHOLDS:
    row_is_high = df["oof_confirmed_probability"] >= threshold

    # Only rows already labeled CONFIRMED can enter the numerator.
    confirmed_high = df.loc[
        (df["label_confirmed"] == 1) & row_is_high,
        ["host_group"]
    ].drop_duplicates()

    screened_hosts = int(confirmed_high["host_group"].nunique())
    screened_fraction = screened_hosts / n_hosts if n_hosts else np.nan

    # Host-level recovery among hosts that have >=1 catalog-confirmed entry.
    confirmed_host_ids = set(
        host_label.loc[
            host_label["has_catalog_confirmed_entry"] == 1, "host_group"
        ]
    )
    recovered_confirmed_hosts = len(
        confirmed_host_ids.intersection(set(confirmed_high["host_group"]))
    )
    host_recall = (
        recovered_confirmed_hosts / len(confirmed_host_ids)
        if confirmed_host_ids else np.nan
    )

    threshold_results.append({
        "threshold": threshold,
        "resolved_host_denominator": n_hosts,
        "baseline_confirmed_hosts": confirmed_hosts,
        "screened_confirmed_hosts": screened_hosts,
        "baseline_catalog_host_fraction": baseline_catalog_host_fraction,
        "screened_catalog_host_fraction": screened_fraction,
        "difference_screened_minus_baseline": screened_fraction - baseline_catalog_host_fraction,
        "confirmed_host_recall": host_recall
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

selected_row = threshold_df.loc[
    threshold_df["threshold"] == SCREENING_THRESHOLD
].iloc[0]

print(
    f"At threshold {SCREENING_THRESHOLD:.2f}, "
    f"{int(selected_row['screened_confirmed_hosts'])} of {n_hosts} "
    f"resolved-sample host stars contribute to the screened catalog-host fraction."
)
print(
    "This is a model-screened catalog diagnostic, not an occurrence-rate estimate."
)

,threshold,resolved_host_denominator,baseline_confirmed_hosts,screened_confirmed_hosts,baseline_catalog_host_fraction,screened_catalog_host_fraction,difference_screened_minus_baseline,confirmed_host_recall
0,0.3,520,353,352,0.678846,0.676923,-0.001923,0.997167
1,0.5,520,353,330,0.678846,0.634615,-0.044231,0.934844
2,0.7,520,353,295,0.678846,0.567308,-0.111538,0.835694


At threshold 0.50, 330 of 520 resolved-sample host stars contribute to the screened catalog-host fraction.
This is a model-screened catalog diagnostic, not an occurrence-rate estimate.


## 8. Export results for reproducibility

The row-level export includes the actual catalog disposition, host identifier, fold, and grouped OOF probability. Summary exports are intended for auditing and comparison, not for claiming a corrected occurrence rate.

In [11]:
# Keep identifiers and key scientific columns when available.
export_cols = [
    col for col in [
        "kepid", "kepoi_name", "koi_disposition",
        "koi_period", "koi_prad", "koi_steff"
    ]
    if col in df.columns
]

oof_export = df[export_cols].copy()
oof_export["label_confirmed"] = y.to_numpy()
oof_export["oof_confirmed_probability"] = oof_probability
oof_export["oof_fold"] = oof_fold
oof_export.to_csv("path3_kdwarf_oof_predictions.csv", index=False)

metrics_df.to_csv("path3_kdwarf_metrics.csv", index=False)
threshold_df.to_csv("path3_kdwarf_thresholds.csv", index=False)
fold_summary.to_csv("path3_kdwarf_fold_summary.csv", index=False)
host_label.to_csv("path3_kdwarf_host_level_labels.csv", index=False)

# Compact compatibility summary with explicitly descriptive names.
compact = pd.DataFrame({
    "metric": [
        "baseline_catalog_host_fraction",
        "screened_catalog_host_fraction_at_0.50",
        "difference_screened_minus_baseline_at_0.50",
        "confirmed_host_recall_at_0.50",
        "resolved_host_denominator",
        "baseline_confirmed_hosts",
        "screened_confirmed_hosts_at_0.50",
        "oof_roc_auc",
        "oof_average_precision"
    ],
    "value": [
        baseline_catalog_host_fraction,
        float(selected_row["screened_catalog_host_fraction"]),
        float(selected_row["difference_screened_minus_baseline"]),
        float(selected_row["confirmed_host_recall"]),
        n_hosts,
        confirmed_hosts,
        int(selected_row["screened_confirmed_hosts"]),
        float(roc_auc_score(y, oof_probability)),
        float(average_precision_score(y, oof_probability))
    ]
})
compact.to_csv("path3_kdwarf_summary.csv", index=False)

print("Saved output files:")
for filename in [
    "path3_kdwarf_summary.csv",
    "path3_kdwarf_metrics.csv",
    "path3_kdwarf_thresholds.csv",
    "path3_kdwarf_fold_summary.csv",
    "path3_kdwarf_host_level_labels.csv",
    "path3_kdwarf_oof_predictions.csv"
]:
    print(" -", filename)

Saved output files:
 - path3_kdwarf_summary.csv
 - path3_kdwarf_metrics.csv
 - path3_kdwarf_thresholds.csv
 - path3_kdwarf_fold_summary.csv
 - path3_kdwarf_host_level_labels.csv
 - path3_kdwarf_oof_predictions.csv


## 9. Interpretation and limitations

1. The K-dwarf temperature interval (3,900–5,200 K) and planet-radius interval (1–4 Earth radii) reproduce the original notebook's filters; they are operational selection cuts, not a complete astrophysical stellar classification.
2. The denominator contains only unique host stars represented by resolved KOI entries in this filtered catalog. It is not the total number of Kepler K-dwarf target stars.
3. The supervised model uses only `CONFIRMED` and `FALSE POSITIVE` entries. Candidates are excluded rather than incorrectly assigned to the false-positive class.
4. GroupKFold keeps every entry for a host star in one fold and produces exactly one held-out prediction per resolved row.
5. The screened catalog-host fraction is a model-based diagnostic applied to already confirmed catalog entries. It is not an independent planet validation and is not corrected for detection completeness, transit geometry, or selection effects.
6. Report the baseline and screened values with their exact denominator and threshold. Do not call their difference a change in the true occurrence rate.

**Recommended ISEF wording:** “We compared the catalog-confirmed host fraction in a restricted KOI K-dwarf/small-planet subset with a threshold-screened diagnostic based on host-star-grouped out-of-fold predictions.”